# 08 - Tabular Data (CSV & Excel) Ingestion

> Complete guide and in-depth Arabic documentation: [`README.md`](./README.md)

Demonstrates basic row loading vs semantic row serialization with metadata filtering.

## 1. Method 1: Row Ingestion with CSVLoader

In [1]:
import os
from pathlib import Path
import pandas as pd
from typing import List, Dict, Any

from langchain_core.documents import Document
from langchain_community.document_loaders import CSVLoader

print(f"Pandas version: {pd.__version__}")
print("LangChain Document Loaders imported successfully!")

C:\Users\aliashraf\AppData\Local\Temp\ipykernel_18564\250397022.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader
d:\Code\AI\rag-mastery\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pandas version: 3.0.6
LangChain Document Loaders imported successfully!


## 2. Method 2: Semantic Row Serialization with Pandas
Converts structured rows into natural language narratives and extracts numeric metadata.

In [2]:
from pathlib import Path
data_dir = Path('data') if Path('data').exists() else Path('../../data')
csv_file = (data_dir / 'tabular' / 'products.csv').resolve()
excel_file = (data_dir / 'tabular' / 'inventory.xlsx').resolve()

print(f"[+] CSV File exists: {csv_file.exists()} (Size: {csv_file.stat().st_size if csv_file.exists() else 0} bytes)")
print(f"[+] Excel File exists: {excel_file.exists()} (Size: {excel_file.stat().st_size if excel_file.exists() else 0} bytes)")

# استعراض سريع لبيانات الـ CSV باستخدام Pandas
df_preview = pd.read_csv(csv_file)
display(df_preview.head())


[+] CSV File exists: True (Size: 541 bytes)
[+] Excel File exists: True (Size: 5857 bytes)


,product,category,price,description
0,MacBook Pro 16,Laptops,2499,High-performance laptop with M3 Max chip and 3...
1,Sony WH-1000XM5,Audio,399,Industry-leading noise cancelling wireless hea...
2,iPhone 15 Pro,Smartphones,1199,Titanium design smartphone with A17 Pro chip a...
3,Dell XPS 15,Laptops,1899,Premium OLED display laptop with Intel Core i7...
4,Logitech MX Master 3S,Accessories,99,Ergonomic wireless mouse with ultra-fast elect...


## 3. Multi-Sheet Excel Ingestion (pd.ExcelFile)
Extracts rows across workbook sheets and tracks sheet names in metadata.

In [3]:
try:
    loader = CSVLoader(
        file_path=str(csv_file),
        encoding="utf-8",
        csv_args={
            "delimiter": ",",
            "quotechar": '"'
        }
    )
    csv_docs = loader.load()

    print(f"[+] Successfully loaded {len(csv_docs)} rows as Documents.\n")
    
    for i, doc in enumerate(csv_docs[:3], 1):
        print(f"--- Document #{i} [Row: {doc.metadata.get('row')}] ---")
        print(f"Content:\n{doc.page_content}")
        print(f"Metadata: {doc.metadata}\n")
except Exception as e:
    print(f"[-] Error loading CSV: {e}")

[+] Successfully loaded 5 rows as Documents.

--- Document #1 [Row: 0] ---
Content:
product: MacBook Pro 16
category: Laptops
price: 2499
description: High-performance laptop with M3 Max chip and 36GB unified memory.
Metadata: {'source': 'data\\products.csv', 'row': 0}

--- Document #2 [Row: 1] ---
Content:
product: Sony WH-1000XM5
category: Audio
price: 399
description: Industry-leading noise cancelling wireless headphones with 30-hour battery life.
Metadata: {'source': 'data\\products.csv', 'row': 1}

--- Document #3 [Row: 2] ---
Content:
product: iPhone 15 Pro
category: Smartphones
price: 1199
description: Titanium design smartphone with A17 Pro chip and advanced camera system.
Metadata: {'source': 'data\\products.csv', 'row': 2}



## 4. Metadata Filtering Simulation for Vector Retrieval

In [4]:
def process_csv_intelligently(file_path: str) -> List[Document]:
    df = pd.read_csv(file_path)
    documents: List[Document] = []

    for idx, row in df.iterrows():
        p_name = row["product"]
        cat = row["category"]
        price = row["price"]
        desc = row["description"]

        # صياغة نص سردي متكامل يفهمه نموذج التضمين بدقة عالية
        content = (
            f"Product Name: {p_name}\n"
            f"Category: {cat}\n"
            f"Retail Price: ${price}\n"
            f"Detailed Description: {desc}"
        )

        # إثراء الميتا-داتا للاستفادة منها في الفلترة بقواعد البيانات المتجهة
        metadata = {
            "source": Path(file_path).name,
            "row_index": int(idx),
            "product_name": str(p_name),
            "category": str(cat),
            "price": float(price),
            "is_premium": float(price) >= 1000.0
        }

        documents.append(Document(page_content=content, metadata=metadata))

    return documents

intelligent_docs = process_csv_intelligently(str(csv_file))
print(f"[+] Generated {len(intelligent_docs)} intelligent documents.\n")

# استعراض أول مستند مع الميتا-داتا الغنية
print("--- Sample Enriched Document ---")
print(f"Content:\n{intelligent_docs[0].page_content}")
print(f"Enriched Metadata:\n{intelligent_docs[0].metadata}")

[+] Generated 5 intelligent documents.

--- Sample Enriched Document ---
Content:
Product Name: MacBook Pro 16
Category: Laptops
Retail Price: $2499
Detailed Description: High-performance laptop with M3 Max chip and 36GB unified memory.
Enriched Metadata:
{'source': 'products.csv', 'row_index': 0, 'product_name': 'MacBook Pro 16', 'category': 'Laptops', 'price': 2499.0, 'is_premium': True}


## 5. Comparison: CSVLoader vs Custom Pandas Serializer

In [5]:
def process_multisheet_excel(excel_path: str) -> List[Document]:
    excel_app = pd.ExcelFile(excel_path)
    sheet_names = excel_app.sheet_names
    print(f"[+] Found {len(sheet_names)} sheets in Excel workbook: {sheet_names}")

    excel_docs: List[Document] = []

    for sheet in sheet_names:
        sheet_df = pd.read_excel(excel_app, sheet_name=sheet)
        
        for idx, row in sheet_df.iterrows():
            # تحويل خلايا الصف إلى تمثيل مهيكل
            row_repr = " | ".join([f"{col}: {val}" for col, val in row.items()])
            
            meta = {
                "source": Path(excel_path).name,
                "sheet_name": sheet,
                "row_index": int(idx),
                "columns_count": len(sheet_df.columns)
            }
            excel_docs.append(Document(page_content=row_repr, metadata=meta))

    return excel_docs

excel_documents = process_multisheet_excel(str(excel_file))
print(f"\n[+] Total rows extracted across all sheets: {len(excel_documents)}\n")

for doc in excel_documents[:4]:
    print(f"[Sheet: {doc.metadata['sheet_name']} | Row {doc.metadata['row_index']}] -> {doc.page_content}")

[+] Found 2 sheets in Excel workbook: ['products', 'summary']

[+] Total rows extracted across all sheets: 9

[Sheet: products | Row 0] -> product: MacBook Pro 16 | category: Laptops | price: 2499 | description: High-performance laptop with M3 Max chip and 36GB unified memory.
[Sheet: products | Row 1] -> product: Sony WH-1000XM5 | category: Audio | price: 399 | description: Industry-leading noise cancelling wireless headphones with 30-hour battery life.
[Sheet: products | Row 2] -> product: iPhone 15 Pro | category: Smartphones | price: 1199 | description: Titanium design smartphone with A17 Pro chip and advanced camera system.
[Sheet: products | Row 3] -> product: Dell XPS 15 | category: Laptops | price: 1899 | description: Premium OLED display laptop with Intel Core i7 and RTX graphics.


## Summary
Semantic serialization significantly improves dense embedding quality for tabular data.